In [1]:
!pip install --upgrade binance-historical-data plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.9 MB/s eta 0:00:00


In [2]:
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta, date, time
from binance_historical_data import BinanceDataDumper
warnings.filterwarnings('ignore')

def get_list_all_trading_pairs():
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",  # spot, um, cm
        data_type="klines",  # aggTrades, klines, trades
        data_frequency="5m",
    )
    return data_dumper.get_list_all_trading_pairs()

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h"):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency= data_frequency,
    )
    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()
    data_dumper.dump_data(
        tickers = ticker,
        date_start = date_start,
        date_end = date_end,
        is_to_update_existing = False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency = "1h"):
    daily_path = os.path.join(os.getcwd(), f"spot/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"spot/monthly/klines/{ticker}/{data_frequency}")
    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)
    all_files = daily_files + monthly_files
    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker}")
        return None
    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)
    return data



In [3]:
ticker = 'BTCUSDT'
data_frequency = '15m'
download_ticker(ticker, "2024-01-01", "2025-07-25", data_frequency)
df = process_csv_files(ticker, data_frequency)
df = df.drop(columns=['close_time',	'quote_asset_volume',	'number_of_trades',	'taker_buy_base_asset_volume',	'taker_buy_quote_asset_volume',	'ignore'])
df.set_index('open_time', inplace=True)
df

---> Found overall tickers: 601
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 15m
---> Start Date: 20240101
---> End Date: 20250725


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/18 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/25 [00:00<?, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 18 months 25 days
---> For BTCUSD new data saved for: 0 months 0 days


,Open,High,Low,Close,volume
open_time,,,,,
2024-01-01 00:15:00,42488.00,42554.57,42412.02,42419.73,392.24889
2024-01-01 00:30:00,42419.73,42447.82,42354.19,42441.32,319.90644
2024-01-01 00:45:00,42441.32,42490.74,42422.45,42475.23,127.81493
2024-01-01 01:00:00,42475.23,42475.23,42431.65,42466.33,188.76099
2024-01-01 01:15:00,42466.32,42513.17,42466.32,42493.16,152.48691
...,...,...,...,...,...
2025-07-25 22:45:00,117340.15,117340.15,117203.81,117279.97,57.32558
2025-07-25 23:00:00,117279.97,117328.92,117186.83,117224.55,83.13741
2025-07-25 23:15:00,117224.55,117357.01,117224.54,117312.00,44.29278


In [4]:
# Calculate moving average for prices
short_window = 7    #Adjust params here
long_window = 28
df['MA_long'] = df['Close'].rolling(window=long_window).mean()
df['MA_short'] = df['Close'].rolling(window=short_window).mean()

# Position
df['Position'] = np.select((
    (df['MA_short'] > df['MA_long']) & (df['MA_short'].shift(1) < df['MA_long'].shift(1)),
    (df['MA_short'] < df['MA_long']) & (df['MA_short'].shift(1) > df['MA_long'].shift(1))),
                           (1, -1),
                           default=0)
df['Position'] = df['Position'].replace(0, method='ffill')

df.head(100)

,Open,High,Low,Close,volume,MA_long,MA_short,Position
open_time,,,,,,,,
2024-01-01 00:15:00,42488.00,42554.57,42412.02,42419.73,392.24889,NaN,NaN,0
2024-01-01 00:30:00,42419.73,42447.82,42354.19,42441.32,319.90644,NaN,NaN,0
2024-01-01 00:45:00,42441.32,42490.74,42422.45,42475.23,127.81493,NaN,NaN,0
2024-01-01 01:00:00,42475.23,42475.23,42431.65,42466.33,188.76099,NaN,NaN,0
2024-01-01 01:15:00,42466.32,42513.17,42466.32,42493.16,152.48691,NaN,NaN,0
...,...,...,...,...,...,...,...,...
2024-01-02 00:00:00,44179.55,44650.00,44148.34,44630.09,2154.59235,43496.449643,43933.622857,1
2024-01-02 00:15:00,44630.10,44844.00,44450.00,44609.79,1997.62342,43563.545357,44099.727143,1
2024-01-02 00:30:00,44609.78,44768.29,44553.92,44732.01,1002.64655,43635.008571,44271.452857,1


In [5]:
def calculate_returns(pos,close):
  entry = [0] * len(pos)
  pnls = [0] * len(pos)
  fee = 0.05/100
  for i in range (1,len(pos)):
    if pos[i] == pos[i-1]:
      entry[i] = entry[i-1]
    elif pos[i] != pos[i-1]:
      entry[i] = close[i]

  for i in range (1,len(pos)):
    if entry[i] != entry[i-1]:
      pnls[i] = (entry[i] - entry[i-1])*pos[i-1]*(1-fee)

  return entry, pnls

df['Price'], df['Pnls'] = calculate_returns(df['Position'], df['Close'])


df.head (500) # Display the DataFrame

,Open,High,Low,Close,volume,MA_long,MA_short,Position,Price,Pnls
open_time,,,,,,,,,,
2024-01-01 00:15:00,42488.00,42554.57,42412.02,42419.73,392.24889,NaN,NaN,0,0.00,0.0
2024-01-01 00:30:00,42419.73,42447.82,42354.19,42441.32,319.90644,NaN,NaN,0,0.00,0.0
2024-01-01 00:45:00,42441.32,42490.74,42422.45,42475.23,127.81493,NaN,NaN,0,0.00,0.0
2024-01-01 01:00:00,42475.23,42475.23,42431.65,42466.33,188.76099,NaN,NaN,0,0.00,0.0
2024-01-01 01:15:00,42466.32,42513.17,42466.32,42493.16,152.48691,NaN,NaN,0,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...
2024-01-06 04:00:00,43931.10,43971.10,43916.32,43923.77,98.31490,43966.761786,43924.220000,-1,43886.06,0.0
2024-01-06 04:15:00,43923.77,43928.11,43868.88,43898.03,113.62119,43970.663929,43920.940000,-1,43886.06,0.0
2024-01-06 04:30:00,43898.03,43932.17,43882.04,43920.16,93.24993,43977.195357,43922.490000,-1,43886.06,0.0


In [6]:
count_positive = (df['Pnls'] > 0).sum()
sum_positive = df[df['Pnls'] > 0]['Pnls'].sum()

count_negative = (df['Pnls'] < 0).sum()
sum_negative = df[df['Pnls'] < 0]['Pnls'].sum()

print(f'Số lệnh lãi: {count_positive}')
print(f'Số lệnh lỗ: {count_negative}')
print(f'Tổng lãi: {sum_positive}')
print(f'Lãi trung bình: {sum_positive/count_positive}')
print(f'Tổng lỗ: {sum_negative}')
print(f'Lỗ trung bình: {sum_negative/count_negative}')

Số lệnh lãi: 864
Số lệnh lỗ: 1736
Tổng lãi: 722485.1267250001
Lãi trung bình: 836.2096374131945
Tổng lỗ: -690172.5111150006
Lỗ trung bình: -397.56481055011557


In [7]:
# Plot profit
# plt.plot(unrealised_dates, np.cumprod(1+np.array(unrealised))-1)
# plt.plot(dates, np.cumprod(1+np.array(pnls))-1)
#plt.plot(dates,np.cumprod(1+np.array(pnls))-1)
# plt.plot(unrlzd_dates,np.cumprod(1+np.array(unrlzd)))
plt.plot(dates, np.cumsum(pnls))
#plt.plot(unrlzd_dates, np.cumsum(unrlzd))

NameError: name 'dates' is not defined